In [3]:
import os
import requests
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")
API_TOKEN = os.getenv("TMDB_READ_ACCESS_TOKEN")

headers = {
    "accept": "application/json",
    "Authorization": f"Bearer {API_TOKEN}"
}

# Testing the alternative domain endpoint
ALT_BASE_URL = "https://api.tmdb.org/3"
TEST_MOVIE_ID = 27205  

print("Testing alternative TMDb endpoint (api.tmdb.org)...")
detail_url = f"{ALT_BASE_URL}/movie/{TEST_MOVIE_ID}"

try:
    response = requests.get(detail_url, headers=headers, timeout=5)
    if response.status_code == 200:
        movie_data = response.json()
        print("Bypass Successful via Alternative Domain!")
        print(f"Title: {movie_data.get('title')}")
        print("\nAvailable Keys in Payload:", list(movie_data.keys()))
    else:
        print(f"Connected to alternative domain but failed. Status: {response.status_code}")
except requests.exceptions.Timeout:
    print("Alternative domain also blocked.")
    print("Execute this PowerShell command as Administrator to change your DNS to Cloudflare (1.1.1.1):")
    print('Set-DnsClientServerAddress -InterfaceAlias "Wi-Fi" -ServerAddresses ("1.1.1.1","8.8.8.8")')

Testing alternative TMDb endpoint (api.tmdb.org)...
Connected to alternative domain but failed. Status: 401


In [5]:
import os
import requests
from dotenv import load_dotenv

# Force reload the .env file with real keys
load_dotenv(dotenv_path="../.env", override=True)

ALT_BASE_URL = "https://api.tmdb.org/3"
TEST_MOVIE_ID = 27205  # Inception
API_KEY = os.getenv("TMDB_API_KEY")

params = {"api_key": API_KEY}

print("Testing alternative domain with your real API key...")
response = requests.get(f"{ALT_BASE_URL}/movie/{TEST_MOVIE_ID}", params=params, timeout=5)

if response.status_code == 200:
    movie_data = response.json()
    print("Authentication Successful!")
    print(f"Movie Title: {movie_data.get('title')}")
    print(f"Release Date: {movie_data.get('release_date')}")
    print(f"Overview: {movie_data.get('overview')}")
else:
    print(f"Failed with Status: {response.status_code}")
    print(response.text)

Testing alternative domain with your real API key...
Authentication Successful!
Movie Title: Inception
Release Date: 2010-07-15
Overview: Cobb, a skilled thief who commits corporate espionage by infiltrating the subconscious of his targets is offered a chance to regain his old life as payment for a task considered to be impossible: "inception", the implantation of another person's idea into a target's subconscious.


In [6]:
import os
import requests
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env", override=True)
ALT_BASE_URL = "https://api.tmdb.org/3"
TEST_MOVIE_ID = 27205  
API_KEY = os.getenv("TMDB_API_KEY")
params = {"api_key": API_KEY}

# 1. Fetch primary details for the poster path
detail_res = requests.get(f"{ALT_BASE_URL}/movie/{TEST_MOVIE_ID}", params=params, timeout=5)
poster_path = detail_res.json().get("poster_path") if detail_res.status_code == 200 else None
full_poster_url = f"https://image.tmdb.org/t/p/w500{poster_path}" if poster_path else "No Poster Available"

print(f"Poster URL: {full_poster_url}\n")

# 2. Fetch credits endpoint
credits_url = f"{ALT_BASE_URL}/movie/{TEST_MOVIE_ID}/credits"
credits_res = requests.get(credits_url, params=params, timeout=5)

if credits_res.status_code == 200:
    credits_data = credits_res.json()
    
    # Extract Cast (Top 4 billed main actors)
    cast_list = credits_data.get("cast", [])
    top_4_actors = [actor.get("name") for actor in cast_list[:4]]
    
    # Extract Crew (Director and Director of Photography)
    crew_list = credits_data.get("crew", [])
    director = None
    cinematographer = None
    
    for member in crew_list:
        if member.get("job") == "Director":
            director = member.get("name")
        elif member.get("job") in ["Director of Photography", "Cinematographer"]:
            cinematographer = member.get("name")
            
    print("--- Extracted Portfolio Metadata ---")
    print(f"Director: {director}")
    print(f"Director of Photography/Cinematographer: {cinematographer}")
    print(f"Top 4 Billed Actors: {', '.join(top_4_actors)}")
else:
    print(f"Failed to fetch credits. Status: {credits_res.status_code}")

Poster URL: https://image.tmdb.org/t/p/w500/xlaY2zyzMfkhk0HSC5VUwzoZPU1.jpg

--- Extracted Portfolio Metadata ---
Director: Christopher Nolan
Director of Photography/Cinematographer: Wally Pfister
Top 4 Billed Actors: Leonardo DiCaprio, Joseph Gordon-Levitt, Ken Watanabe, Tom Hardy


In [7]:
import sys
import asyncio
# Append src directory context to the path
sys.path.append("../")
from src.tmdb_client import TMDBClient

async def test_production_client():
    client = TMDBClient()
    
    print("1. Testing Async Movie Search ('Inception')...")
    search_results = await client.search_movie("Inception")
    if search_results:
        target_id = search_results[0]["id"]
        print(f"   Found Movie ID: {target_id}")
        
        print("\n2. Testing Parallel Metadata Compilation...")
        meta = await client.get_movie_metadata(target_id)
        if meta:
            print(f"   Title: {meta['title']}")
            print(f"   Director: {meta['director']}")
            print(f"   Cinematographer: {meta['cinematographer']}")
            print(f"   Cast: {meta['cast']}")
            
        print("\n3. Testing Async Review Streaming Extraction...")
        reviews = await client.get_movie_reviews(target_id)
        print(f"   Extracted {len(reviews)} production reviews.")
    else:
        print("Search failed or returned empty payload.")

# Execute async loop in notebook environment
await test_production_client()

1. Testing Async Movie Search ('Inception')...
   Found Movie ID: 27205

2. Testing Parallel Metadata Compilation...
   Title: Inception
   Director: Christopher Nolan
   Cinematographer: Wally Pfister
   Cast: ['Leonardo DiCaprio', 'Joseph Gordon-Levitt', 'Ken Watanabe', 'Tom Hardy']

3. Testing Async Review Streaming Extraction...
   Extracted 8 production reviews.


In [9]:
import sys
sys.path.append("../")
from src.tmdb_client import TMDBClient
# Extension .py dropped to allow correct module parsing
from src.data_preprocessing import TextPreprocessor

async def verify_preprocessing_pipeline():
    client = TMDBClient()
    processor = TextPreprocessor()
    
    # Fetch live data
    reviews = await client.get_movie_reviews(27205)
    
    print(f"--- Processing {len(reviews)} Live Reviews --- \n")
    for i, rev in enumerate(reviews[:3]):
        raw_text = rev["content"]
        raw_rating = rev["rating"]
        
        clean_text = processor.clean_text(raw_text)
        mapped_label = processor.map_tmdb_rating_to_3class(raw_rating)
        
        label_map = {0: "Negative", 1: "Neutral", 2: "Positive", -1: "Unrated/Mixed"}
        
        print(f"Review #{i+1}:")
        print(f"  Author: {rev['author']}")
        print(f"  Raw Rating: {raw_rating} -> Target Class: {label_map[mapped_label]}")
        print(f"  Raw Snippet: {raw_text[:80]}...")
        print(f"  Clean Snippet: {clean_text[:80]}...\n")

await verify_preprocessing_pipeline()

--- Processing 8 Live Reviews --- 

Review #1:
  Author: ohlalipop
  Raw Rating: None -> Target Class: Unrated/Mixed
  Raw Snippet: When I first saw the trailer for this film, I knew that this would attract a lot...
  Clean Snippet: when i first saw the trailer for this film, i knew that this would attract a lot...

Review #2:
  Author: tmdb44006625
  Raw Rating: 8.0 -> Target Class: Positive
  Raw Snippet: Whether you watch Inception as a heist movie, a redemption story, or a sci-fi ac...
  Clean Snippet: whether you watch inception as a heist movie, a redemption story, or a sci-fi ac...

Review #3:
  Author: Varya
  Raw Rating: None -> Target Class: Unrated/Mixed
  Raw Snippet: Is there anyone on earth who doesn't like Christopher Nolan’s films? Inception i...
  Clean Snippet: is there anyone on earth who doesn't like christopher nolan’s films? inception i...



In [1]:
import sys
sys.path.append("../")
from src.model_utils import ModelInferencePlatform

def test_inference_engines():
    platform = ModelInferencePlatform()
    
    print("1. Loading Engine 1 (Local 3-Class Baseline)...")
    if platform.load_baseline_engine():
        print("   Engine 1 Loaded Successfully!")
    else:
        print("   Failed to load Engine 1. Check your .pkl file path.")
        
    print("\n2. Loading Engine 2 (Remote HF Binary DistilBERT)...")
    print("   (This might take a moment to stream the weights if it's the first run...)")
    if platform.load_legacy_hf_engine():
        print("   Engine 2 Loaded Successfully from Hugging Face Hub!")
    else:
        print("   Failed to stream Engine 2 from Hugging Face.")
        return

    # Head-to-Head test cases
    sample_reviews = [
        "An absolute masterpiece! Best film of the decade.",
        "The movie was completely average. It had okay parts but dragged heavily."
    ]
    
    print("\n--- Running Head-to-Head Test Inference ---")
    for i, review in enumerate(sample_reviews):
        print(f"\nTest Review #{i+1}: '{review}'")
        
        # Engine 1 Prediction
        res_b = platform.predict_baseline(review)
        print(f"  -> {res_b['engine']} Label: {res_b['label']} (Confidence: {res_b['confidence']:.4f})")
        
        # Engine 2 Prediction
        res_hf = platform.predict_legacy_hf(review)
        print(f"  -> {res_hf['engine']} Label: {res_hf['label']} (Confidence: {res_hf['confidence']:.4f})")

test_inference_engines()

1. Loading Engine 1 (Local 3-Class Baseline)...
   Engine 1 Loaded Successfully!

2. Loading Engine 2 (Remote HF Binary DistilBERT)...
   (This might take a moment to stream the weights if it's the first run...)


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

   Engine 2 Loaded Successfully from Hugging Face Hub!

--- Running Head-to-Head Test Inference ---

Test Review #1: 'An absolute masterpiece! Best film of the decade.'
  -> Engine 1 (Traditional Baseline) Label: Positive (Confidence: 0.7177)
  -> Engine 2 (Binary Legacy Anchor) Label: LABEL_1 (Confidence: 0.9855)

Test Review #2: 'The movie was completely average. It had okay parts but dragged heavily.'
  -> Engine 1 (Traditional Baseline) Label: Neutral (Confidence: 0.9443)
  -> Engine 2 (Binary Legacy Anchor) Label: LABEL_0 (Confidence: 0.9867)


In [1]:
import sys
import importlib
# Append src directory context to the path
sys.path.append("../")

import src.model_utils
# Force Python to bypass the cache and read the file fresh off the disk
importlib.reload(src.model_utils)

from src.model_utils import ModelInferencePlatform

def verify_triple_engine_platform():
    platform = ModelInferencePlatform()
    
    print("-> Initializing complete machine learning framework...")
    platform.load_baseline_engine()
    platform.load_legacy_hf_engine()
    platform.load_transformer_3class_engine()
    print("   All 3 Engines loaded and initialized successfully!\n")
    
    test_phrase = "The movie was mediocre. Fine acting but a highly standard plot."
    print(f"Test Review: '{test_phrase}'\n")
    
    # 3-Way Head-to-Head Inference Execution
    for pred_func in [platform.predict_baseline, platform.predict_legacy_hf, platform.predict_transformer_3class]:
        res = pred_func(test_phrase)
        print(f"  * {res['engine']}")
        print(f"    Prediction: {res['label']} | Confidence: {res['confidence']:.4f}\n")

verify_triple_engine_platform()

-> Initializing complete machine learning framework...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

   All 3 Engines loaded and initialized successfully!

Test Review: 'The movie was mediocre. Fine acting but a highly standard plot.'

  * Engine 1 (Traditional Baseline)
    Prediction: Neutral | Confidence: 0.8944

  * Engine 2 (Binary Legacy Anchor)
    Prediction: LABEL_0 | Confidence: 0.7969

  * Engine 3 (3-Class Fine-Tuned DistilBERT)
    Prediction: Neutral | Confidence: 0.6336



In [1]:
import os
from pathlib import Path

def inspect_project_directory(root_dir):
    root_path = Path(root_dir)
    print(f"📁 Scanning Project Repository: {root_path.resolve()}\n" + "="*60)
    
    # Exclude system folders from cluttering the terminal output
    ignored_folders = {'.git', '__pycache__', '.pytest_cache', 'venv', '.env'}
    
    for root, dirs, files in os.walk(root_path):
        dirs[:] = [d for d in dirs if d not in ignored_folders]
        
        current_path = Path(root)
        depth = len(current_path.relative_to(root_path).parts)
        indent = "    " * depth
        
        if current_path != root_path:
            print(f"{indent}└── 📂 {current_path.name}/")
            
        file_indent = "    " * (depth + 1)
        for file in files:
            file_absolute = current_path / file
            file_size_kb = file_absolute.stat().st_size / 1024
            print(f"{file_indent}├── 📄 {file} ({file_size_kb:.1f} KB)")

# Executes the scanner on your current workspace directory
inspect_project_directory(os.getcwd())

📁 Scanning Project Repository: F:\Portfolio Projects\tmdb_sentiment_pipeline\notebooks
    ├── 📄 01_api_exploration.ipynb (18.8 KB)
    ├── 📄 02_3class_baseline.ipynb (89.1 KB)
    ├── 📄 03_3class_transformer.ipynb (135.1 KB)
